# MERC Ablation Study

Systematically removes/replaces architectural modules to measure each component's contribution.

**Ablated components:**
- `full` — baseline (Path A + Path B + cross-modal attention + speaker embedding)
- `no_path_a` — remove hypergraph path (only spectral filtering)
- `no_path_b` — remove spectral path (only hypergraph)
- `no_paths` — remove both paths (Stage 2 → MLP directly)
- `no_cross_modal` — remove cross-modal attention inside Path A
- `no_speaker_emb` — remove speaker embeddings (sinusoidal PE only)
- `d=32` — smaller hidden dim
- `d=128` — larger hidden dim (original)

Results saved to `ablation_results/`.

In [1]:
import os, sys, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Ensure correct working directory
REPO_ROOT = '/mnt/Work/ML/Thesis/BMVC/Hopeful'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.config import MERCConfig
from src.run_iemocap import main as run_iemocap
from src.run_meld import main as run_meld

ABLATION_DIR = 'ablation_results'
os.makedirs(ABLATION_DIR, exist_ok=True)
print(f'Output → {ABLATION_DIR}/')

Output → ablation_results/


## Ablation configurations

Adjust `ABLATION_EPOCHS` and `IEMOCAP_K_FOLDS` for speed vs. reliability tradeoff.

In [2]:
ABLATION_EPOCHS    = 25   # reduce for quick runs; use 60 for final results
IEMOCAP_K_FOLDS   = 3    # 3 is faster; use 5 for final results
BATCH_SIZE         = 8
LR                 = 1e-3
SEED               = 42

# Each entry: (display_name, iemocap_overrides, meld_overrides)
# Overrides are merged on top of dataset defaults
ABLATION_CONFIGS = [
    ('full',           {},                                          {}                                         ),
    ('no_path_a',      {'use_path_a': False},                       {'use_path_a': False}                      ),
    ('no_path_b',      {'use_path_b': False},                       {'use_path_b': False}                      ),
    ('no_paths',       {'use_path_a': False, 'use_path_b': False},  {'use_path_a': False, 'use_path_b': False} ),
    ('no_cross_modal', {'use_cross_modal_attn': False},             {'use_cross_modal_attn': False}            ),
    ('no_speaker_emb', {'use_speaker_emb': False},                  {'use_speaker_emb': False}                 ),
    ('d=32',           {'d': 32, 'd_spk': 16},                      {'d': 32, 'd_spk': 16}                    ),
    ('d=128',          {'d': 128, 'd_spk': 64},                     {'d': 128, 'd_spk': 64}                   ),
]

BASE_KWARGS = dict(epochs=ABLATION_EPOCHS, batch_size=BATCH_SIZE, lr=LR, seed=SEED)

print(f'Configs: {[c[0] for c in ABLATION_CONFIGS]}')
print(f'IEMOCAP: {IEMOCAP_K_FOLDS} folds × {ABLATION_EPOCHS} epochs × {len(ABLATION_CONFIGS)} configs')
print(f'MELD:    1 run × {ABLATION_EPOCHS} epochs × {len(ABLATION_CONFIGS)} configs')

Configs: ['full', 'no_path_a', 'no_path_b', 'no_paths', 'no_cross_modal', 'no_speaker_emb', 'd=32', 'd=128']
IEMOCAP: 3 folds × 25 epochs × 8 configs
MELD:    1 run × 25 epochs × 8 configs


## IEMOCAP ablations

Session 5 = test, Sessions 1–4 = trainval, k-fold CV for dev.

In [3]:
iemocap_results = {}

for name, ie_overrides, _ in ABLATION_CONFIGS:
    print(f"\n{'='*55}")
    print(f"  IEMOCAP — {name}")
    print(f"{'='*55}")
    try:
        cfg = MERCConfig(
            dataset='iemocap',
            iemocap_k_folds=IEMOCAP_K_FOLDS,
            **BASE_KWARGS,
            **ie_overrides,
        )
        out          = run_iemocap(cfg)
        fold_results = out['fold_results']
        run_dir      = out['run_dir']

        wa_vals  = [r['wa']  for r in fold_results]
        wf1_vals = [r['wf1'] for r in fold_results]
        uf1_vals = [r['uf1'] for r in fold_results]

        iemocap_results[name] = {
            'wa_mean':  float(np.mean(wa_vals)),
            'wf1_mean': float(np.mean(wf1_vals)),
            'uf1_mean': float(np.mean(uf1_vals)),
            'wa_std':   float(np.std(wa_vals)),
            'wf1_std':  float(np.std(wf1_vals)),
            'uf1_std':  float(np.std(uf1_vals)),
            'fold_results': fold_results,
            'run_dir': run_dir,
        }
        r = iemocap_results[name]
        print(f"  WA  {r['wa_mean']:.4f} ± {r['wa_std']:.4f}")
        print(f"  WF1 {r['wf1_mean']:.4f} ± {r['wf1_std']:.4f}")
        print(f"  UF1 {r['uf1_mean']:.4f} ± {r['uf1_std']:.4f}")
        print(f"  run → {run_dir}")

    except Exception as e:
        print(f"  ERROR: {e}")
        iemocap_results[name] = {'error': str(e)}

print("\nIEMOCAP ablations complete.")


  IEMOCAP — full
Run directory: results/iemocap_20260526_171236
Loading IEMOCAP features…
  trainval: 120 conversations  |  test (Session5): 31
  trainable parameters: 377,463

  Fold 0  train=80 dev=40 convs
    Ep   TrLoss   DevLoss   DevWF1   DevUF1    TeWF1
     0   1.8576    1.7040   0.1756   0.1602   0.1523
    10   0.0790    0.1048   0.5224   0.5028   0.5423
    20   0.0569    0.0802   0.5875   0.5832   0.5391
    24   0.0543    0.0897   0.5846   0.5763   0.5483
  [plot] results/iemocap_20260526_171236/plots/iemocap_fold0_curves.png
  [fold 0] WA 0.5475  WF1 0.5391  UF1 0.5295  (best ep 20)

  Fold 1  train=80 dev=40 convs
    Ep   TrLoss   DevLoss   DevWF1   DevUF1    TeWF1
     0   1.9227    1.6903   0.1461   0.0942   0.1563
    10   0.0805    0.0872   0.5544   0.5294   0.4938
    20   0.0586    0.0844   0.5526   0.5333   0.4488
    24   0.0512    0.0850   0.5984   0.5699   0.4994
  [plot] results/iemocap_20260526_171236/plots/iemocap_fold1_curves.png
  [fold 1] WA 0.4982  WF

## MELD ablations

Official train/dev/test splits. Dev used for checkpoint selection.

In [4]:
meld_results = {}

for name, _, meld_overrides in ABLATION_CONFIGS:
    print(f"\n{'='*55}")
    print(f"  MELD — {name}")
    print(f"{'='*55}")
    try:
        cfg = MERCConfig(
            dataset='meld',
            **BASE_KWARGS,
            **meld_overrides,
        )
        out = run_meld(cfg)
        te  = out['te_final']

        meld_results[name] = {
            'wa':          float(te['wa']),
            'wf1':         float(te['wf1']),
            'uf1':         float(te['uf1']),
            'f1_per_class': [float(x) for x in te.get('f1_per_class', [])],
            'run_dir':     out['run_dir'],
        }
        r = meld_results[name]
        print(f"  WA  {r['wa']:.4f}")
        print(f"  WF1 {r['wf1']:.4f}")
        print(f"  UF1 {r['uf1']:.4f}")
        print(f"  run → {r['run_dir']}")

    except Exception as e:
        print(f"  ERROR: {e}")
        meld_results[name] = {'error': str(e)}

print("\nMELD ablations complete.")


  MELD — full
Run directory: results/meld_20260526_172530
Building MELD speaker map from train split…
  speakers: 113 (112 regular + 1 background)
Loading MELD datasets…
  train: 968  dev: 108  test: 268 convs
  trainable parameters: 381,133

  Ep   TrLoss   DevLoss   DevWF1   DevUF1    TeWF1
--------------------------------------------------------
   0   1.5834    1.5462   0.2513   0.0849   0.3126
   1   1.4784    1.5168   0.4199   0.2500   0.4308
   2   1.4175    1.4244   0.4383   0.2642   0.4614
   3   1.3758    1.4016   0.4457   0.2808   0.4681
   4   1.3435    1.4464   0.4168   0.2380   0.4522
   5   1.3206    1.3803   0.4674   0.3008   0.4757
   6   0.9456    1.0159   0.4700   0.3005   0.4825
   7   0.6615    0.7214   0.4806   0.3145   0.4832
   8   0.4174    0.4770   0.4685   0.2987   0.4795
   9   0.2268    0.2771   0.4904   0.3321   0.4936
  10   0.0641    0.0845   0.4392   0.3376   0.4478
  11   0.0598    0.0822   0.4228   0.3284   0.4400
  12   0.0576    0.0832   0.4117   0

## Comparison tables

In [5]:
MELD_EMOTIONS = ['neutral', 'joy', 'surprise', 'anger', 'sadness', 'disgust', 'fear']
IEMOCAP_EMOTIONS = ['hap', 'sad', 'neu', 'ang', 'exc', 'fru']

# --- IEMOCAP table ---
ie_rows = []
for name, r in iemocap_results.items():
    if 'error' in r:
        ie_rows.append({'config': name, 'WA': 'ERROR', 'WF1': 'ERROR', 'UF1': 'ERROR'})
    else:
        ie_rows.append({
            'config': name,
            'WA':  f"{r['wa_mean']:.4f} ± {r['wa_std']:.4f}",
            'WF1': f"{r['wf1_mean']:.4f} ± {r['wf1_std']:.4f}",
            'UF1': f"{r['uf1_mean']:.4f} ± {r['uf1_std']:.4f}",
            'wa_mean': r['wa_mean'], 'wf1_mean': r['wf1_mean'], 'uf1_mean': r['uf1_mean'],
        })

df_ie = pd.DataFrame(ie_rows).set_index('config')
print('IEMOCAP (Session-5 test, mean ± std across folds):')
display(df_ie[['WA', 'WF1', 'UF1']])

# --- MELD table ---
meld_rows = []
for name, r in meld_results.items():
    if 'error' in r:
        meld_rows.append({'config': name, 'WA': 'ERROR', 'WF1': 'ERROR', 'UF1': 'ERROR'})
    else:
        meld_rows.append({'config': name, 'WA': r['wa'], 'WF1': r['wf1'], 'UF1': r['uf1']})

df_meld = pd.DataFrame(meld_rows).set_index('config')
print('\nMELD (official test set):')
display(df_meld.round(4))

IEMOCAP (Session-5 test, mean ± std across folds):


,WA,WF1,UF1
config,,,
full,0.5284 ± 0.0216,0.5300 ± 0.0222,0.5197 ± 0.0183
no_path_a,0.2263 ± 0.0426,0.2023 ± 0.0568,0.2106 ± 0.0412
no_path_b,0.5376 ± 0.0317,0.5392 ± 0.0324,0.5271 ± 0.0307
no_paths,0.5031 ± 0.0131,0.4993 ± 0.0027,0.4897 ± 0.0044
no_cross_modal,0.5775 ± 0.0226,0.5722 ± 0.0163,0.5486 ± 0.0139
no_speaker_emb,ERROR,ERROR,ERROR
d=32,0.5454 ± 0.0196,0.5469 ± 0.0143,0.5252 ± 0.0163
d=128,0.5771 ± 0.0063,0.5721 ± 0.0075,0.5539 ± 0.0146



MELD (official test set):


,WA,WF1,UF1
config,,,
full,0.5154,0.4936,0.3011
no_path_a,0.4804,0.3508,0.1339
no_path_b,0.5158,0.4770,0.2750
no_paths,0.5181,0.4872,0.2880
no_cross_modal,0.5196,0.4992,0.3135
no_speaker_emb,0.5235,0.4974,0.3080
d=32,0.5081,0.4759,0.2769
d=128,0.5096,0.4761,0.2725


## Save results

In [6]:
# CSVs
df_ie.to_csv(f'{ABLATION_DIR}/iemocap_ablation.csv')
df_meld.to_csv(f'{ABLATION_DIR}/meld_ablation.csv')

# Per-class F1 for MELD
meld_pc_rows = []
for name, r in meld_results.items():
    if 'error' not in r and r.get('f1_per_class'):
        row = {'config': name}
        row.update(dict(zip(MELD_EMOTIONS, r['f1_per_class'])))
        meld_pc_rows.append(row)
if meld_pc_rows:
    pd.DataFrame(meld_pc_rows).set_index('config').round(4).to_csv(
        f'{ABLATION_DIR}/meld_ablation_perclass.csv'
    )

# Per-class F1 for IEMOCAP
ie_pc_rows = []
for name, r in iemocap_results.items():
    if 'error' not in r:
        frs = r['fold_results']
        if frs and frs[0].get('f1_per_class'):
            n_cls = len(frs[0]['f1_per_class'])
            mean_cls = [float(np.mean([fr['f1_per_class'][c] for fr in frs])) for c in range(n_cls)]
            row = {'config': name}
            row.update(dict(zip(IEMOCAP_EMOTIONS, mean_cls)))
            ie_pc_rows.append(row)
if ie_pc_rows:
    pd.DataFrame(ie_pc_rows).set_index('config').round(4).to_csv(
        f'{ABLATION_DIR}/iemocap_ablation_perclass.csv'
    )

# Full JSON summary (without large fold_results lists)
def _serialisable(r):
    return {k: v for k, v in r.items() if k not in ('fold_results',)}

summary = {
    'settings': {'epochs': ABLATION_EPOCHS, 'k_folds': IEMOCAP_K_FOLDS, 'seed': SEED},
    'iemocap': {k: _serialisable(v) for k, v in iemocap_results.items()},
    'meld':    {k: _serialisable(v) for k, v in meld_results.items()},
}
with open(f'{ABLATION_DIR}/ablation_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Saved:')
for fn in os.listdir(ABLATION_DIR):
    print(f'  {ABLATION_DIR}/{fn}')

Saved:
  ablation_results/ablation_summary.json
  ablation_results/iemocap_ablation.csv
  ablation_results/iemocap_ablation_perclass.csv
  ablation_results/meld_ablation.csv
  ablation_results/meld_ablation_perclass.csv


## Comparison plots

In [7]:
COLORS = {
    'full':           '#1565C0',
    'no_path_a':      '#EF5350',
    'no_path_b':      '#FF7043',
    'no_paths':       '#B71C1C',
    'no_cross_modal': '#AB47BC',
    'no_speaker_emb': '#26A69A',
    'd=32':           '#78909C',
    'd=128':          '#42A5F5',
}
DEFAULT_COLOR = '#90CAF9'


def _bar_ablation(ax, configs, vals, errs, ylabel, title):
    x = np.arange(len(configs))
    colors = [COLORS.get(c, DEFAULT_COLOR) for c in configs]
    bars = ax.bar(x, vals, yerr=errs, color=colors, capsize=4,
                  edgecolor='white', linewidth=0.5, error_kw={'elinewidth': 1.2})
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.003,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(configs, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ymin = max(0.0, min(vals) - 0.06)
    ymax = min(1.0, max(vals) + 0.06)
    ax.set_ylim(ymin, ymax)
    ax.axhline(vals[0], color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


# --- IEMOCAP WF1 + UF1 side-by-side ---
valid_ie = [(n, r) for n, r in iemocap_results.items() if 'error' not in r]
if valid_ie:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    names = [n for n, _ in valid_ie]

    _bar_ablation(
        axes[0], names,
        [r['wf1_mean'] for _, r in valid_ie],
        [r['wf1_std']  for _, r in valid_ie],
        'WF1 (weighted F1)', 'IEMOCAP — Weighted F1'
    )
    _bar_ablation(
        axes[1], names,
        [r['uf1_mean'] for _, r in valid_ie],
        [r['uf1_std']  for _, r in valid_ie],
        'UF1 (macro F1)', 'IEMOCAP — Macro F1'
    )
    fig.suptitle(
        f'IEMOCAP ablation  (Session-5 test, {IEMOCAP_K_FOLDS}-fold mean±std, {ABLATION_EPOCHS} ep)',
        fontsize=12
    )
    plt.tight_layout()
    save_path = f'{ABLATION_DIR}/iemocap_ablation.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved {save_path}')

# --- MELD WF1 + UF1 side-by-side ---
valid_meld = [(n, r) for n, r in meld_results.items() if 'error' not in r]
if valid_meld:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    names = [n for n, _ in valid_meld]

    _bar_ablation(
        axes[0], names,
        [r['wf1'] for _, r in valid_meld],
        [0.0] * len(valid_meld),
        'WF1 (weighted F1)', 'MELD — Weighted F1'
    )
    _bar_ablation(
        axes[1], names,
        [r['uf1'] for _, r in valid_meld],
        [0.0] * len(valid_meld),
        'UF1 (macro F1)', 'MELD — Macro F1'
    )
    fig.suptitle(f'MELD ablation  (official test set, {ABLATION_EPOCHS} ep)', fontsize=12)
    plt.tight_layout()
    save_path = f'{ABLATION_DIR}/meld_ablation.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved {save_path}')

Saved ablation_results/iemocap_ablation.png
Saved ablation_results/meld_ablation.png


## Per-class F1 heatmap

In [8]:
def _perclass_heatmap(results_dict, emotion_names, key_fn, title, save_path):
    valid = [(n, r) for n, r in results_dict.items()
             if 'error' not in r and key_fn(r)]
    if not valid:
        print('No per-class data available.')
        return

    configs = [n for n, _ in valid]
    matrix  = np.array([key_fn(r) for _, r in valid])  # (C, n_emotions)

    fig, ax = plt.subplots(figsize=(len(emotion_names) * 1.1 + 1, len(configs) * 0.6 + 1.5))
    im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn', vmin=0.0, vmax=1.0)
    ax.set_xticks(range(len(emotion_names)))
    ax.set_xticklabels(emotion_names, fontsize=9)
    ax.set_yticks(range(len(configs)))
    ax.set_yticklabels(configs, fontsize=9)
    for i in range(len(configs)):
        for j in range(len(emotion_names)):
            ax.text(j, i, f'{matrix[i, j]:.2f}', ha='center', va='center',
                    fontsize=8, color='black' if 0.3 < matrix[i, j] < 0.8 else 'white')
    ax.set_title(title, fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.03)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved {save_path}')


# IEMOCAP per-class F1 heatmap (mean across folds)
_perclass_heatmap(
    iemocap_results,
    IEMOCAP_EMOTIONS,
    key_fn=lambda r: [
        float(np.mean([fr['f1_per_class'][c] for fr in r['fold_results']]))
        for c in range(len(IEMOCAP_EMOTIONS))
    ] if r.get('fold_results') and r['fold_results'][0].get('f1_per_class') else None,
    title='IEMOCAP per-class F1 by ablation config',
    save_path=f'{ABLATION_DIR}/iemocap_perclass_heatmap.png',
)

# MELD per-class F1 heatmap
_perclass_heatmap(
    meld_results,
    MELD_EMOTIONS,
    key_fn=lambda r: r.get('f1_per_class') or None,
    title='MELD per-class F1 by ablation config',
    save_path=f'{ABLATION_DIR}/meld_perclass_heatmap.png',
)

Saved ablation_results/iemocap_perclass_heatmap.png
Saved ablation_results/meld_perclass_heatmap.png


## Delta from full model

In [9]:
def _delta_table(results_dict, metric_key, baseline_key='full', label='WF1 Δ vs full'):
    baseline = results_dict.get(baseline_key)
    if not baseline or 'error' in baseline:
        print('Baseline not available.')
        return None

    base_val = baseline[metric_key]
    rows = []
    for name, r in results_dict.items():
        if 'error' in r:
            rows.append({'config': name, label: 'ERROR'})
        else:
            delta = r[metric_key] - base_val
            rows.append({'config': name, label: f'{delta:+.4f}'})
    df = pd.DataFrame(rows).set_index('config')
    return df


print('IEMOCAP — WF1 delta vs full model:')
df_ie_delta = _delta_table(iemocap_results, 'wf1_mean')
if df_ie_delta is not None:
    display(df_ie_delta)

print('\nMELD — WF1 delta vs full model:')
df_meld_delta = _delta_table(meld_results, 'wf1')
if df_meld_delta is not None:
    display(df_meld_delta)

# Save delta tables
if df_ie_delta is not None:
    df_ie_delta.to_csv(f'{ABLATION_DIR}/iemocap_delta.csv')
if df_meld_delta is not None:
    df_meld_delta.to_csv(f'{ABLATION_DIR}/meld_delta.csv')
print(f'\nAll ablation results in {ABLATION_DIR}/')

IEMOCAP — WF1 delta vs full model:


,WF1 Δ vs full
config,
full,+0.0000
no_path_a,-0.3277
no_path_b,+0.0092
no_paths,-0.0307
no_cross_modal,+0.0422
no_speaker_emb,ERROR
d=32,+0.0169
d=128,+0.0421



MELD — WF1 delta vs full model:


,WF1 Δ vs full
config,
full,+0.0000
no_path_a,-0.1428
no_path_b,-0.0166
no_paths,-0.0064
no_cross_modal,+0.0056
no_speaker_emb,+0.0038
d=32,-0.0177
d=128,-0.0176



All ablation results in ablation_results/
